In [26]:
import agama
import gc_utils
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from sklearn.svm import SVC

agama.setUnits(mass=1, length=1, velocity=1)
units = agama.getUnits()

In [27]:
pot_mw_fil = "../.venv312/lib/python3.12/site-packages/agama/data/MWPotential2014.ini"
pot_mw = agama.Potential(pot_mw_fil)

In [28]:
r_lim = 3
xyz = np.array([r_lim, r_lim * 0, r_lim * 0])
vy = np.sqrt(-r_lim * pot_mw.force(xyz)[0])
vxyz = np.array([0, vy, 0])
posvel = np.concatenate((xyz, vxyz))  # gc orbit

In [29]:
mass_gc = 1e5
r_gc = 0.004  # radius of gc

time_total = 2  # in time units (0.978 Gyr)
num_particles = int(1e4)  # number of particles in the stream

In [30]:
N = 100

In [31]:
time_init = time_total
time_gc, orbit_gc = agama.orbit(potential=pot_mw, ic=posvel, time=time_init, trajsize=N + 1)

In [32]:
pot_gc = agama.Potential(type="Plummer", mass=mass_gc, scaleRadius=r_gc)

In [33]:
df_gc = agama.DistributionFunction(type="QuasiSpherical", potential=pot_gc)

In [34]:
xv, mass = agama.GalaxyModel(pot_gc, df_gc).sample(num_particles)

In [35]:
print("velocity dispersion in the progenitor: %g km/s" % np.mean(xv[:, 3:6] ** 2 / 3) ** 0.5)

velocity dispersion in the progenitor: 1.88237 km/s


In [36]:
xv += orbit_gc[0]

In [37]:
time_begin = 0  # start of the simulation
time_end = time_total
time_index = 0

while time_begin < time_end:
    time_index = min(time_index + 10, len(time_gc) - 1)
    pot_gc_moving = agama.Potential(potential=pot_gc, center=np.column_stack([time_gc, orbit_gc]))
    pot_total = agama.Potential(pot_mw, pot_gc_moving)

    xv = np.vstack(
        agama.orbit(
            ic=xv,
            potential=pot_total,
            trajsize=1,
            time=time_gc[time_index] - time_begin,
            timestart=time_begin,
            verbose=False,
            accuracy=1e-6,
        )[:, 1]
    )

    time_begin = time_gc[time_index]
    xv_rel = xv - orbit_gc[time_index]

    pot_gc = agama.Potential(type="multipole", particles=(xv_rel[:, 0:3], mass), symmetry="s")
    bound = pot_gc.potential(xv_rel[:, 0:3]) + 0.5 * np.sum(xv_rel[:, 3:6] ** 2, axis=1) < 0

    print(
        "t=%6.3f, Phi(0)=%6.2f, bound mass=%5.0f"
        % (time_begin, pot_gc.potential(0, 0, 0), np.sum(mass[bound]))
    )

t= 0.200, Phi(0)=-35.20, bound mass=17426
t= 0.400, Phi(0)=  -inf, bound mass=   50
t= 0.600, Phi(0)=  -inf, bound mass=    0
t= 0.800, Phi(0)= -0.25, bound mass=    0
t= 1.000, Phi(0)= -0.19, bound mass=    0
t= 1.200, Phi(0)= -0.15, bound mass=    0
t= 1.400, Phi(0)= -0.13, bound mass=    0
t= 1.600, Phi(0)= -0.12, bound mass=    0
t= 1.800, Phi(0)= -0.11, bound mass=    0
t= 2.000, Phi(0)= -0.10, bound mass=    0


Okayyyyyy I can now use GC details to get projected mass loss rates.